# import library

In [1]:
!pip install -q underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.6/978.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 65.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.2 which is incompatible.


In [2]:
import pandas as pd
import os
import re
import string
from underthesea import word_tokenize, text_normalize
import random
from sklearn.model_selection import train_test_split

In [3]:
root_dir = "/kaggle/input/data-nlp-bai-2/data"
df_1 = pd.DataFrame()
for path in os.listdir(root_dir):
  df = pd.read_csv(os.path.join(root_dir, path))
  df_1 = pd.concat([df, df_1])
df_1

,image_path,description,question,answer
0,12006.jpg,"Bức ảnh hiển thị một phần tài liệu khoa học, c...","Theo bạn, thí nghiệm này có thể được áp dụng t...",Thí nghiệm này có thể được áp dụng trong thực ...
1,12007.jpg,"Bức ảnh hiển thị một bảng mục lục với 3 cột, m...",Bài học nào được liệt kê trong tuần 20?,Tuần 20 bao gồm các bài học sau: \n- Đọc thơ\...
2,12007.jpg,"Bức ảnh hiển thị một bảng mục lục với 3 cột, m...","Bài ""Luyện tập về thao tác lập luận bác bỏ"" xu...","Bài ""Luyện tập về thao tác lập luận bác bỏ"" xu..."
3,12007.jpg,"Bức ảnh hiển thị một bảng mục lục với 3 cột, m...","Bài đọc thêm ""Tổng biệt hành"" nằm trong tuần nào?","Bài đọc thêm ""Tổng biệt hành"" nằm trong tuần 23."
4,12007.jpg,"Bức ảnh hiển thị một bảng mục lục với 3 cột, m...","Bài ""Tráng giang"" của tác giả nào?","Bài ""Tráng giang"" là của tác giả Huy Cận."
...,...,...,...,...
19091,51855.jpg,Bức ảnh bao gồm hai phần chính. Phần trên là c...,Câu hỏi 1 đề cập đến vấn đề gì?,Câu hỏi 1 đề cập đến khả năng lưu trữ văn bản ...
19092,51855.jpg,Bức ảnh bao gồm hai phần chính. Phần trên là c...,Câu hỏi 2 đề cập đến các biện pháp bảo vệ dữ l...,Câu hỏi 2 đề cập đến 5 biện pháp bảo vệ dữ liệ...
19093,51855.jpg,Bức ảnh bao gồm hai phần chính. Phần trên là c...,Hình ảnh minh họa cho câu hỏi 1 là gì?,Hình ảnh minh họa cho câu hỏi 1 là 4 khung tho...
19094,51855.jpg,Bức ảnh bao gồm hai phần chính. Phần trên là c...,Hình ảnh minh họa cho phần bài tập vận dụng là...,Hình ảnh minh họa cho phần bài tập vận dụng là...


In [4]:
df_1.shape

(259096, 4)

# data processing

In [5]:
def processing_df(df, root_dir):
    df = df.rename({"image_path": "image_id"}, axis = 1)
    df["image_path"] = df['image_id'].apply(lambda x: os.path.join(root_dir, x))
    df["image_id"] = df["image_id"].str.split(".").str[0]
    df = df.dropna().copy()
    df = df[df['image_path'] != '/kaggle/input/data-nlp-bai-2/img/img/6803.jpg']
    df = df[df['image_path'] != '/kaggle/input/data-nlp-bai-2/img/img/6801.jpg']
    df['question'] = df['question'].astype(str)
    df['answer'] = df['answer'].astype(str)
    df.drop(['description'], axis = 1, inplace = True)
    df.drop_duplicates(inplace = True, ignore_index = True)
    df.reset_index(inplace=True)
    return df

In [6]:
df = processing_df(df_1, '/kaggle/input/data-nlp-bai-2/img/img')
# df = df.iloc[:100]
df

,index,image_id,question,answer,image_path
0,0,12006,"Theo bạn, thí nghiệm này có thể được áp dụng t...",Thí nghiệm này có thể được áp dụng trong thực ...,/kaggle/input/data-nlp-bai-2/img/img/12006.jpg
1,1,12007,Bài học nào được liệt kê trong tuần 20?,Tuần 20 bao gồm các bài học sau: \n- Đọc thơ\...,/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
2,2,12007,"Bài ""Luyện tập về thao tác lập luận bác bỏ"" xu...","Bài ""Luyện tập về thao tác lập luận bác bỏ"" xu...",/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
3,3,12007,"Bài đọc thêm ""Tổng biệt hành"" nằm trong tuần nào?","Bài đọc thêm ""Tổng biệt hành"" nằm trong tuần 23.",/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
4,4,12007,"Bài ""Tráng giang"" của tác giả nào?","Bài ""Tráng giang"" là của tác giả Huy Cận.",/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
...,...,...,...,...,...
259055,259055,51855,Câu hỏi 1 đề cập đến vấn đề gì?,Câu hỏi 1 đề cập đến khả năng lưu trữ văn bản ...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259056,259056,51855,Câu hỏi 2 đề cập đến các biện pháp bảo vệ dữ l...,Câu hỏi 2 đề cập đến 5 biện pháp bảo vệ dữ liệ...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259057,259057,51855,Hình ảnh minh họa cho câu hỏi 1 là gì?,Hình ảnh minh họa cho câu hỏi 1 là 4 khung tho...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259058,259058,51855,Hình ảnh minh họa cho phần bài tập vận dụng là...,Hình ảnh minh họa cho phần bài tập vận dụng là...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg


In [7]:
print("NaN trong df:")
print(df.isna().sum())

NaN trong df:
index         0
image_id      0
question      0
answer        0
image_path    0
dtype: int64


In [8]:
def normalize_text(text):
    clean_text = text.lower()

    replacements = {
        "nxb": "nhà xuất bản",
    }

    for old, new in replacements.items():
        clean_text = clean_text.replace(old, new)

    # Xóa ký tự đặc biệt VD (&nbsp;)
    clean_text = re.sub(r'&\S+;', ' ', clean_text)

    # Xóa emoji
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"
                               u"\U0001F300-\U0001F5FF"
                               u"\U0001F680-\U0001F6FF"
                               u"\U0001F1E0-\U0001F1FF"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u'\U00010000-\U0010ffff'
                               u"\u200d"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\u3030"
                               u"\ufe0f"
                               "]+", flags=re.UNICODE)
    clean_text = re.sub(emoji_pattern, " ", clean_text)

    # Xóa ký tự đặc biệt không gõ được
    special_chars = ["“", "”", "…", "•", "–", "’", " ️", "✅", "✓", " ̂́ ̃ ̣̂ ", "‘", "●", "‒", "➤",
                     "★", "ღ", "✪", "‎", "➦", "×", "", "✿", "☆", "◤", "◕", "❁", "‿",
                     "❀", "■", "█", "☛", "⑴⒪⑵⑵⑴⑺", "►", "°", "»", "ø", "➽", "", "✧", "✽", "*",
                     "➫", "【", "】", "⇒", "卐", "♛", "±", "∞", "②", "⑥", "①", "⑦", "➋", "➊", "➌",
                     "✓", "™", "®", "", ""]
    for char in special_chars:
        clean_text = clean_text.replace(char, " ")

    # Bỏ các ký tự đặc biệt (dấu câu)
    clean_text = ''.join(' ' if char in string.punctuation else char for char in clean_text)

    # Loại bỏ khoảng trắng thừa
    clean_text = re.sub(r"\s+", " ", clean_text)
    clean_text = re.sub(r"^[\s]", "", clean_text)
    clean_text = re.sub(r"[\s]$", "", clean_text)

    # Đảm bảo dấu ở đúng chữ (ví dụ: oà, uý)
    clean_text = text_normalize(clean_text)

    # Tách câu thành từ
    clean_text = word_tokenize(clean_text, format="text")

    return clean_text

def apply_normalize_text_to_dataframe(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: normalize_text(str(x)))
    return df

In [9]:
columns_to_process = ['question', 'answer']
df = apply_normalize_text_to_dataframe(df, columns_to_process)
df = apply_normalize_text_to_dataframe(df, columns_to_process)
df

,index,image_id,question,answer,image_path
0,0,12006,theo bạn thí_nghiệm này có_thể được áp_dụng tr...,thí_nghiệm này có_thể được áp_dụng trong thực_...,/kaggle/input/data-nlp-bai-2/img/img/12006.jpg
1,1,12007,bài_học nào được liệt_kê trong tuần 20,tuần 20 bao_gồm các bài_học sau đọc thơ_nghĩa ...,/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
2,2,12007,bài luyện_tập về thao_tác lập_luận bác_bỏ xuất...,bài luyện_tập về thao_tác lập_luận bác_bỏ xuất...,/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
3,3,12007,bài đọc thêm tổng_biệt_hành nằm trong tuần nào,bài đọc thêm tổng_biệt_hành nằm trong tuần 23,/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
4,4,12007,bài tráng_giang của tác_giả nào,bài tráng_giang là của tác_giả huy_cận,/kaggle/input/data-nlp-bai-2/img/img/12007.jpg
...,...,...,...,...,...
259055,259055,51855,câu hỏi 1 đề_cập đến vấn_đề gì,câu hỏi 1 đề_cập đến khả_năng lưu_trữ văn_bản ...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259056,259056,51855,câu hỏi 2 đề_cập đến các biện_pháp bảo_vệ dữ_l...,câu hỏi 2 đề_cập đến 5 biện_pháp bảo_vệ dữ_liệ...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259057,259057,51855,hình_ảnh minh_họa cho câu hỏi 1 là gì,hình_ảnh minh_họa cho câu hỏi 1 là 4 khung_tho...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg
259058,259058,51855,hình_ảnh minh_họa cho phần bài_tập vận_dụng là gì,hình_ảnh minh_họa cho phần bài_tập vận_dụng là...,/kaggle/input/data-nlp-bai-2/img/img/51855.jpg


In [10]:
samples = df.sample(n=random.randint(2, min(2, len(df)))).reset_index(drop=True)

for i, row in samples.iterrows():
    image = row['image_path']
    question = row['question']
    answer = row['answer']

    try:
        # Mở và hiển thị hình ảnh
        img = Image.open(image)

        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"Câu hỏi: {question}\nCâu trả lời: {answer}", fontsize=10)
        plt.axis('off')  # Ẩn trục để hiển thị gọn gàng hơn
        plt.show()
    except FileNotFoundError:
        print(f"Không tìm thấy hình ảnh: {image}")
    except Exception as e:
        print(f"Lỗi khi tải hình ảnh {image}: {e}")

Lỗi khi tải hình ảnh /kaggle/input/data-nlp-bai-2/img/img/46887.jpg: name 'Image' is not defined
Lỗi khi tải hình ảnh /kaggle/input/data-nlp-bai-2/img/img/28011.jpg: name 'Image' is not defined


In [11]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Kiểm tra kích thước
print("Number of rows in train set:", len(df_train))
print("Number of rows in test set:", len(df_test))
print("Train set columns:", df_train.columns)
print("Test set columns:", df_test.columns)

Number of rows in train set: 207248
Number of rows in test set: 51812
Train set columns: Index(['index', 'image_id', 'question', 'answer', 'image_path'], dtype='object')
Test set columns: Index(['index', 'image_id', 'question', 'answer', 'image_path'], dtype='object')


In [12]:
df_train.to_csv("df_train.csv", index=False)

In [13]:
df_test.to_csv("df_test.csv", index=False)